# Streamlit Deployment from Jupyter Notebook

This notebook helps you deploy your completed Sentinel-2 classification results using **Streamlit**.

It does **not retrain** Random Forest, SVM, TabNet or FT-Transformer.  
It uses the results you already produced:

- `all_model_metrics.csv`
- `all_area_statistics.csv`
- `all_feature_importance_statistics.csv`
- `class_code_mapping.csv`
- confusion matrix PNGs
- classified map PNG previews

The notebook can:

1. Unzip the ready-made Streamlit app package.
2. Check whether the required files exist.
3. Run the Streamlit app locally from Jupyter.
4. Create a clean zip folder that can be uploaded to GitHub.
5. Help you prepare the app for Streamlit Community Cloud.

The Streamlit app shows:

- model performance comparison
- confusion matrices
- classified map previews
- area statistics
- feature importance
- CSV upload for computing NDVI, NDBI, MNDWI, BSI and IBI

## 1. Install required packages

Run this cell first.

These packages are light and stable. This first deployment version does **not** require `rasterio`, `torch`, `pytorch-tabnet` or saved `.joblib` model files, because it is a results dashboard.

In [1]:
!pip install streamlit pandas numpy plotly pillow

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
# !pip show anyio
# !pip show jupyter-server

In [ ]:
#!pip install "anyio<4"

In [ ]:
#!pip install anyio==3.7.1

## 2. Set your app paths

Put the Streamlit zip file in the same folder as this notebook, or edit `APP_ZIP`.

If you downloaded the zip file from ChatGPT, the file name should be:

```text
sentinel2_lulc_streamlit_app.zip
```

In [2]:
from pathlib import Path
import zipfile
import shutil
import os
import sys
import subprocess
import time
import webbrowser
import pandas as pd

# # # Folder where this notebook is running
NOTEBOOK_FOLDER = Path.cwd()

# # # Change this only if your zip file is somewhere else
# # # APP_ZIP = NOTEBOOK_FOLDER / "sentinel2_lulc_streamlit_app.zip"

# # # This is where the app folder will be created
APP_DIR = NOTEBOOK_FOLDER #/ "sentinel2_lulc_streamlit_app"

# print("Notebook folder:", NOTEBOOK_FOLDER)
# print("Expected zip file:", APP_ZIP)
# print("App folder:", APP_DIR)

## 3. Unzip the ready-made Streamlit app package

Run this cell if you already have the zip package.

It will create:

```text
sentinel2_lulc_streamlit_app/
├── app.py
├── requirements.txt
├── README.md
├── data/
├── images/
└── metadata/
```

In [ ]:
# if APP_ZIP.exists():
#     if APP_DIR.exists():
#         print("Existing app folder found:", APP_DIR)
#         print("It will be replaced with a fresh copy from the zip.")
#         shutil.rmtree(APP_DIR)

#     with zipfile.ZipFile(APP_ZIP, "r") as zf:
#         zf.extractall(NOTEBOOK_FOLDER)

#     print("App package extracted successfully.")
#     print("Created folder:", APP_DIR)
# else:
#     print("Zip file not found.")
#     print("Expected location:", APP_ZIP)
#     print("If you do not have the zip, run the next section to create the app files from this notebook.")

## 4. Optional: create the app files directly from this notebook

Use this section only if you do **not** have the zip file.

This will create the app folder and write `app.py` and `requirements.txt`.  
After running it, you must copy your `data` and `images` folders into the app folder.

In [3]:
from pathlib import Path
import json

CREATE_APP_FROM_NOTEBOOK = False  # Change to True only if you do not have the zip file

APP_PY_CODE = ''
REQUIREMENTS_TEXT = 'streamlit\npandas\nnumpy\nplotly\npillow\n'

if CREATE_APP_FROM_NOTEBOOK:
    APP_DIR.mkdir(parents=True, exist_ok=True)
    (APP_DIR / "data").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "images" / "confusion_matrices").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "images" / "classified_maps").mkdir(parents=True, exist_ok=True)
    (APP_DIR / "metadata").mkdir(parents=True, exist_ok=True)

    (APP_DIR / "app.py").write_text(APP_PY_CODE, encoding="utf-8")
    (APP_DIR / "requirements.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")

    readme_text = '''
# Sentinel-2 LULC Streamlit Dashboard

Run locally with:

streamlit run app.py

Main files required:
- data/all_model_metrics.csv
- data/all_area_statistics.csv
- data/all_feature_importance_statistics.csv
- data/class_code_mapping.csv
- images/confusion_matrices/
- images/classified_maps/
'''
    (APP_DIR / "README.md").write_text(readme_text.strip(), encoding="utf-8")

    metadata = {
        "project": "Sentinel-2 LULC classification Streamlit dashboard",
        "note": "Created from Jupyter notebook. Copy your CSV and PNG results into the data and images folders."
    }
    (APP_DIR / "metadata" / "metadata.json").write_text(json.dumps(metadata, indent=4), encoding="utf-8")

    print("App files created at:", APP_DIR)
else:
    print("Skipped. Set CREATE_APP_FROM_NOTEBOOK = True only if you do not have the zip package.")

Skipped. Set CREATE_APP_FROM_NOTEBOOK = True only if you do not have the zip package.


## 5. Check that your app files are present

This checks the files needed by the dashboard.

The app can still run if one or two images are missing, but the main CSV files should be present.

In [4]:
required_files = [
    APP_DIR / "app.py",
    APP_DIR / "requirements.txt",
    APP_DIR / "data" / "all_model_metrics.csv",
    APP_DIR / "data" / "all_area_statistics.csv",
    APP_DIR / "data" / "all_feature_importance_statistics.csv",
    APP_DIR / "data" / "class_code_mapping.csv",
]

check_rows = []
for file in required_files:
    check_rows.append({
        "file": str(file.relative_to(APP_DIR)) if APP_DIR in file.parents else str(file),
        "exists": file.exists()
    })

check_df = pd.DataFrame(check_rows)
display(check_df)

confusion_dir = APP_DIR / "images" / "confusion_matrices"
maps_dir = APP_DIR / "images" / "classified_maps"

confusion_count = len(list(confusion_dir.glob("*.png"))) if confusion_dir.exists() else 0
map_count = len(list(maps_dir.glob("*.png"))) if maps_dir.exists() else 0

print("Confusion matrix PNG count:", confusion_count)
print("Classified map PNG count:", map_count)

if not (APP_DIR / "app.py").exists():
    raise FileNotFoundError("app.py was not found. Extract the zip or create the app files first.")

,file,exists
0,app.py,True
1,requirements.txt,True
2,data\all_model_metrics.csv,True
3,data\all_area_statistics.csv,True
4,data\all_feature_importance_statistics.csv,True
5,data\class_code_mapping.csv,True


Confusion matrix PNG count: 12
Classified map PNG count: 12


## 6. Preview the CSV tables inside Jupyter

This confirms that Streamlit will be able to read your results.

In [5]:
data_files = {
    "Metrics": APP_DIR / "data" / "all_model_metrics.csv",
    "Area statistics": APP_DIR / "data" / "all_area_statistics.csv",
    "Feature importance": APP_DIR / "data" / "all_feature_importance_statistics.csv",
    "Class mapping": APP_DIR / "data" / "class_code_mapping.csv",
}

for name, path in data_files.items():
    print("\n" + "=" * 70)
    print(name)
    print(path)
    if path.exists():
        df = pd.read_csv(path)
        print("Shape:", df.shape)
        display(df.head())
    else:
        print("Missing")


Metrics
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\data\all_model_metrics.csv
Shape: (12, 10)


,Year,Model,Accuracy,Balanced Accuracy,Precision,Recall,F1-Score,MCC,Mean IoU,Source
0,2017,FT-Transformer,0.99,0.98,0.99,0.98,0.98,0.98,0.97,derived from uploaded confusion matrix images
1,2017,Random Forest,0.98,0.97,0.99,0.97,0.98,0.98,0.96,derived from uploaded confusion matrix images
2,2017,SVM,0.98,0.97,0.98,0.97,0.98,0.97,0.95,derived from uploaded confusion matrix images
3,2017,TabNet,0.99,0.99,0.99,0.99,0.99,0.99,0.98,derived from uploaded confusion matrix images
4,2021,FT-Transformer,0.97,0.97,0.96,0.97,0.97,0.96,0.94,derived from uploaded confusion matrix images



Area statistics
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\data\all_area_statistics.csv
Shape: (60, 8)


,Year,Model,map_code,original_class_label,pixel_count,area_m2,area_ha,area_km2
0,2017,RF,1,BL,10257,1025700,102.6,1.0
1,2017,RF,2,BU,477855,47785500,4778.6,47.8
2,2017,RF,3,GL,93549,9354900,935.5,9.4
3,2017,RF,4,TR,149584,14958400,1495.8,15.0
4,2017,RF,5,WB,15222,1522200,152.2,1.5



Feature importance
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\data\all_feature_importance_statistics.csv
Shape: (132, 4)


,Year,Model_Display,feature,importance
0,2017,RF,NDVI,0.14
1,2017,RF,B04,0.13
2,2017,RF,B11,0.13
3,2017,RF,B08,0.11
4,2017,RF,B03,0.10



Class mapping
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\data\class_code_mapping.csv
Shape: (5, 3)


,map_code,encoded_label,original_class_label
0,1,0,Bare land
1,2,1,Built up
2,3,2,Grassland
3,4,3,Tree
4,5,4,Waterbody


## 7. Run the Streamlit app locally from Jupyter

This starts the app on your computer.

After running the cell, open:

```text
http://localhost:8501
```

or click the URL printed in the output.

In [ ]:
# for _ in range(100):
#     line = streamlit_process.stdout.readline()
#     if not line:
#         break
#     print(line, end="")

In [ ]:
# import socket

# sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
# result = sock.connect_ex(("localhost", 8501))

# if result == 0:
#     print("Port 8501 is open")
# else:
#     print("Port 8501 is NOT open")

In [7]:
PORT = 8501

# Stop previous Streamlit process if one is already running from this notebook
try:
    streamlit_process.terminate()
    time.sleep(2)
except Exception:
    pass

cmd = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    str(APP_DIR / "app.py"),
    "--server.port",
    str(PORT),
    "--server.headless",
    "false"
]

print("Running command:")
print(" ".join(cmd))

streamlit_process = subprocess.Popen(
    cmd,
    cwd=str(APP_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

local_url = f"http://localhost:{PORT}"
print("If the browser does not open automatically, open this link:")
print(local_url)

try:
    webbrowser.open(local_url)
except Exception:
    pass

print("\nStreamlit has started in the background.")
print("Run the next cell if you want to see recent log output.")

Running command:
C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe -m streamlit run C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\app.py --server.port 8501 --server.headless false
If the browser does not open automatically, open this link:
http://localhost:8501

Streamlit has started in the background.
Run the next cell if you want to see recent log output.


## 8. Show recent Streamlit log output

Run this if the app does not open.

In [ ]:
# try:
#     # Read a few lines without blocking forever.
#     for _ in range(20):
#         line = streamlit_process.stdout.readline()
#         if not line:
#             break
#         print(line.rstrip())
# except Exception as e:
#     print("Could not read Streamlit logs:", e)

## 9. Stop the Streamlit app

Run this cell when you are finished testing locally.

In [ ]:
# try:
#     streamlit_process.terminate()
#     print("Streamlit app stopped.")
# except Exception as e:
#     print("No running Streamlit process found or could not stop it:", e)

## 10. Create a clean zip for GitHub upload

After the app works locally, run this cell.  
It creates a zip file that you can upload to GitHub or keep as your deployment backup.

In [ ]:
DEPLOY_ZIP = NOTEBOOK_FOLDER / "sentinel2_lulc_streamlit_app_for_github.zip"

if DEPLOY_ZIP.exists():
    DEPLOY_ZIP.unlink()

with zipfile.ZipFile(DEPLOY_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in APP_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, file_path.relative_to(APP_DIR.parent))

print("Created deployment zip:")
print(DEPLOY_ZIP)

## 11. How to publish online

Once the app works locally:

1. Create a new GitHub repository.
2. Upload the contents of the `sentinel2_lulc_streamlit_app` folder.
3. Make sure `app.py` and `requirements.txt` are in the repository.
4. Open Streamlit Community Cloud.
5. Connect your GitHub repository.
6. Select `app.py` as the main file.
7. Deploy the app and copy the public link.

Important:

- Do not upload very large `.tif` raster files to GitHub for this first version.
- This deployment shows your results through CSV tables and PNG previews.
- Live model prediction can be added later after the dashboard link works.

## 12. Report wording

You can adapt this wording in your report:

> The classification outputs were deployed through an interactive Streamlit dashboard. The dashboard presents the outputs from Random Forest, RBF-SVM, TabNet and FT-Transformer, including model performance metrics, confusion matrices, classified map previews, area statistics and feature importance. A CSV upload page was also included to calculate the same Sentinel-2 spectral indices used during the classification workflow.

This wording is safe because it describes your actual implementation rather than claiming ownership of external tools or algorithms.

In [ ]:
# from pathlib import Path
# import sys
# import subprocess
# import time
# import webbrowser
# import socket

# # Change this to the folder where app.py is located
# APP_DIR = Path(r"C:/Users/simba.jombo/OneDrive - Nottingham Trent University/Desktop/Streamlit/Data_streamlit/sentinel2_lulc_streamlit_app")

# # Check that app.py exists
# app_file = APP_DIR / "app.py"

# if not app_file.exists():
#     raise FileNotFoundError(
#         f"Cannot find app.py here:\n{app_file}\n\n"
#         "Check that you unzipped the Streamlit app folder correctly."
#     )

# # Function to check whether a port is already being used
# def port_is_busy(port):
#     with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
#         return s.connect_ex(("127.0.0.1", port)) == 0

# # Use port 8501, or 8502 if 8501 is already busy
# PORT = 8501

# if port_is_busy(PORT):
#     print(f"Port {PORT} is already busy. Using port 8502 instead.")
#     PORT = 8502

# # Save Streamlit logs so that errors are visible
# log_path = APP_DIR / "streamlit_log.txt"
# log_file = open(log_path, "w", encoding="utf-8")

# cmd = [
#     sys.executable,
#     "-m",
#     "streamlit",
#     "run",
#     "app.py",
#     "--server.port",
#     str(PORT),
#     "--server.headless",
#     "false"
# ]

# print("Starting Streamlit with this command:")
# print(" ".join(cmd))

# streamlit_process = subprocess.Popen(
#     cmd,
#     cwd=str(APP_DIR),
#     stdout=log_file,
#     stderr=subprocess.STDOUT
# )

# time.sleep(10)
# log_file.flush()

# # Check whether the app is still running
# if streamlit_process.poll() is None:
#     url = f"http://127.0.0.1:{PORT}"
#     print("Streamlit is running.")
#     print("Open this link:")
#     print(url)
#     webbrowser.open(url)
# else:
#     print("Streamlit stopped because there is an error.")
#     print("Open this log file to see the error:")
#     print(log_path)
#     print("\nLast log lines:")
#     print(log_path.read_text(encoding="utf-8")[-3000:])

In [9]:
from pathlib import Path
import sys
import subprocess
import time
import webbrowser
import socket
import os

# ============================================================
# 1. SET YOUR APP FOLDER
# ============================================================

# APP_DIR = Path(
#     r"C:\Users\simba.jombo\OneDrive - Nottingham Trent University\Desktop\Streamlit\Data_streamlit\sentinel2_lulc_streamlit_app\sentinel2_lulc_streamlit_app"
# )

# APP_DIR = Path(
#     r"C:\Users\N1386471\OneDrive - Nottingham Trent University\Desktop\Streamlit\Data_streamlit\sentinel2_lulc_streamlit_app\sentinel2_lulc_streamlit_app"
# )

APP_DIR = Path(
    r"C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU"
)

APP_FILE = APP_DIR / "app.py"

if not APP_FILE.exists():
    raise FileNotFoundError(
        f"app.py was not found here:\n{APP_FILE}\n\n"
        "Check that APP_DIR points to the folder that contains app.py."
    )

print("App folder found:")
print(APP_DIR)


# ============================================================
# 2. CREATE STREAMLIT CONFIG TO SKIP EMAIL PROMPT
# ============================================================

streamlit_config_dir = APP_DIR / ".streamlit"
streamlit_config_dir.mkdir(parents=True, exist_ok=True)

config_file = streamlit_config_dir / "config.toml"

config_file.write_text(
    """
[server]
headless = true
showEmailPrompt = false
port = 8501

[browser]
gatherUsageStats = false
""".strip(),
    encoding="utf-8"
)

print("Streamlit config written to:")
print(config_file)


# ============================================================
# 3. ALSO CREATE GLOBAL STREAMLIT CREDENTIALS FILE
# ============================================================
# This prevents Streamlit from asking for an email address on first run.

home_streamlit_dir = Path.home() / ".streamlit"
home_streamlit_dir.mkdir(parents=True, exist_ok=True)

credentials_file = home_streamlit_dir / "credentials.toml"

credentials_file.write_text(
    """
[general]
email = ""
""".strip(),
    encoding="utf-8"
)

print("Streamlit credentials written to:")
print(credentials_file)


# ============================================================
# 4. CHECK WHETHER PORT IS BUSY
# ============================================================

def port_is_busy(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

PORT = 8501

if port_is_busy(PORT):
    print(f"Port {PORT} is already busy. Using port 8502 instead.")
    PORT = 8502


# ============================================================
# 5. START STREAMLIT
# ============================================================

log_path = APP_DIR / "streamlit_log.txt"

# Close old process if it exists
try:
    streamlit_process.terminate()
    time.sleep(2)
except Exception:
    pass

env = os.environ.copy()
env["STREAMLIT_SERVER_HEADLESS"] = "true"
env["STREAMLIT_BROWSER_GATHER_USAGE_STATS"] = "false"
env["STREAMLIT_SERVER_SHOW_EMAIL_PROMPT"] = "false"

cmd = [
    sys.executable,
    "-m",
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    str(PORT),
    "--server.headless",
    "true",
    "--server.showEmailPrompt",
    "false",
    "--browser.gatherUsageStats",
    "false"
]

print("\nStarting Streamlit with this command:")
print(" ".join(cmd))

log_file = open(log_path, "w", encoding="utf-8")

streamlit_process = subprocess.Popen(
    cmd,
    cwd=str(APP_DIR),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
    text=True,
    env=env
)

time.sleep(8)
log_file.flush()

# ============================================================
# 6. CHECK WHETHER IT STARTED
# ============================================================

if streamlit_process.poll() is None:
    url = f"http://127.0.0.1:{PORT}"
    print("\nStreamlit is running successfully.")
    print("Open this link:")
    print(url)

    try:
        webbrowser.open(url)
    except Exception:
        pass

else:
    print("\nStreamlit stopped because there is still an error.")
    print("Open this log file:")
    print(log_path)

    try:
        print("\nLast log lines:")
        print(log_path.read_text(encoding="utf-8")[-4000:])
    except Exception as e:
        print("Could not read log file:", e)

App folder found:
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU
Streamlit config written to:
C:\Users\simba.jombo\Desktop\Streamlit\LULC_Classification_NTU\.streamlit\config.toml
Streamlit credentials written to:
C:\Users\simba.jombo\.streamlit\credentials.toml

Starting Streamlit with this command:
C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe -m streamlit run app.py --server.port 8501 --server.headless true --server.showEmailPrompt false --browser.gatherUsageStats false

Streamlit is running successfully.
Open this link:
http://127.0.0.1:8501
